## SargassoDB

Set up a database, just with sample information (V1V2, V4, metabolites)\
KL 5 April 2026\
#back up and uncomment out all the cells here...useful for troubleshooting as I add new datasources\
KL 6 April 2026\
Now try this with actual data...most of which seems to be in BIOS-SCOPE DNA Master.2026.03.10.xlsx

In [10]:
%reset -f
#%whos #also useful at times

In [11]:
import pandas as pd
import os
import pdb
from ftplib import FTP
from tqdm import tqdm

#need this to see the full column width
pd.set_option('display.max_colwidth', None)

In [12]:
def trimSuffix(one):
    one = one.removesuffix('.gz').removesuffix('.fastq')    
    return one

In [13]:
# #now I see why Ben was deleting the database...otherwise get multiple inserts
# but I cannot get this to work as it is still in use and I am having trouble closing it.
# %tried;
# session.close()
# engine.dispose()
# def delete_db():
#     print('Deleting database')
#     db_path = 'new_database.db'
#     if os.path.exists(db_path):
# #         os.unlink(db_path)
#         os.remove(db_path)

In [14]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from sqlalchemy import update,select
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_testing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

# create a session factory
Session = sessionmaker(bind=engine)

# create a declarative base
Base = declarative_base()

In [15]:
# define the classes

class DiscreteInfo(Base):
    __tablename__ = 'discrete'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cruise = Column(String)
    cast = Column(String)
    niskin = Column(String)
    yyyymmdd = Column(String)
    nominalDepth = Column(String)
    V1V2data = Column(String)
    V4data = Column(String)
    mtabData = Column(String)
    
    #need this next row to get the nice output (other get a generic thing ?: <__main__.DiscreteInfo object at 0x000001A3FC7A0F70>)
    def __repr__(self):
        return f"<DiscreteInfo(bottleID='{self.bottleID}', cruise='{self.cruise}')>"


class SeqInfoV1V2(Base):
    __tablename__ = 'sequencingV1V2'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V1V2data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self):
        return f"<SeqInfoV1V2(bottleID='{self.bottleID}', filename='{self.filename}', V1V2data ='{self.V1V2data}')>"
    
class SeqInfoV4(Base):
    __tablename__ = 'sequencingV4'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V4data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"SeqInfoV4(id={self.id!r}, name={self.bottleID!r}, filename={self.filename!r})"
  
   
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    source = Column(String)
    def __repr__(self) -> str:
        return f"index(id={self.id!r}, filename={self.filename!r})"
    
    
class MetaboliteInfo(Base):
    __tablename__ = 'metabolites'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    dataSource = Column(String)
    
    def __repr__(self) -> str:
        return f"MetaboliteInfo(id={self.id!r}, bottleID={self.bottleID!r}, dataSource={self.dataSource!r})"
    
class MtabUntargetedInfo(Base):
    __tablename__ = 'metabolitesUntargeted'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    dataSource = Column(String)
    
    def __repr__(self) -> str:
        return f"MtabUntargetedInfo(id={self.id!r}, bottleID={self.bottleID!r}, dataSource={self.dataSource!r})"

In [16]:
# create the database tables
Base.metadata.create_all(engine)

In [17]:
# # insert some data, setup functions, one per data type
def load_discrete_info():
    print('Loading discrete sample information')
    data_dir = '../test_data/'
    fName = 'BATS_BS_COMBINED_MASTER_mini.xlsx'
    #fName = 'BATS_BS_COMBINED_MASTER_latest.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name='DATA'))

    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = DiscreteInfo()
        db.bottleID = row['New_ID'] 
        db.cruise = row['Cruise_ID']
        db.cast = row['Cast']
        db.niskin = row['Niskin']
        db.yyyymmdd = row['yyyymmdd']
        db.nominalDepth = row['Nominal_Depth']
        session.add(db)
    
    session.commit()
    
def load_V416S_sequencing_info():
    print('Loading V416S sequencing information')
    data_dir = '../test_data/'
    #fName = 'V4_dada2_read_info_03052026.xlsx'
    fName = 'BIOS-SCOPE DNA Master 2026.03.10.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),header=0,
                                   sheet_name = 'BIOS-SCOPE Samples 2014-2023',
                                   dtype={'New_Bottle_ID':str,'Cruise':str,'Cast':str,'Depths':str}))
           
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = SeqInfoV4()
        #pdb.set_trace()
        db.bottleID = row['New_Bottle_ID'] 
        db.cast = row['Cast']
        db.filename = row['V4_16s_Sequencing_File']
        db.V4data = fName
        session.add(db)
    
    session.commit()
    
def load_V1V2_sequencing_info():
    print('Loading V1V2 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V1V2_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),
                                   dtype={'Bottle ID':str,'Cruise':str,'Cast':str,'Depth':str}))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !
    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = SeqInfoV1V2()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FileName']
        db.V1V2data = fName
        session.add(db)
    
    session.commit()

def load_cyverse_info():
    print('Loading list of sequencing files')
    dataDir = '../test_data/Luis_fileLists/' #separate files have already been concatenated
    fName = 'filelist_concatenated.csv'
    df = pd.read_csv(os.path.join(dataDir,fName))

#     #strip off the end of the filename (no...sometimes they appear in DNA sequence master)
#     for index,row in df.iterrows():
#         file = os.path.basename(row.to_string()).strip('.gz')
#         df.loc[index,'filename'] = file

    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = CyverseInfo()
        db.filename = row['filename'] 
        db.source = row['source']
        session.add(db)
    
    session.commit()

def load_metabolite_info():
    print("Loading metabolite information from MetaboLights")
    dataDir = '../test_data/'
    # start with one dataset at MetaboLights --> MTBLS2356 is Longnecker et al.
    study_id = 'MTBLS2356'
    ftp = FTP('ftp.ebi.ac.uk') #address from MetaboLights webpage
    ftp.login()
    ftpDataAddress = '/pub/databases/metabolights/studies/public/' + study_id
    ftp.cwd(ftpDataAddress)
    fileList = ftp.nlst() #can use this to make a list that will be searchable
    
    #start with the metadata about the samples 
    str = 's_' + study_id #this is the search string for the data files
    metadataFiles = [v for v in fileList if str in v] 
    metadataFiles = pd.DataFrame(metadataFiles,columns = ['files'])
    readFile = metadataFiles.loc[0,'files']

    #pdb.set_trace()
    # metadataFiles: put them here 
    # Is there a way to download an FTP file and not write it disk?
    #writeFile = 'dataDir/' + 'tempMetadata.txt'
    writeFile = os.path.join(dataDir,'tempMetadata.txt')

    with open(writeFile,'wb') as fp:
        try:
            retr_command = f"RETR {readFile}"
            ftp.retrbinary(retr_command, fp.write)
        except Exception as e: 
            print(f"Error during quit: {e}")
        except AttributeError as e: 
            print(f"AttributeError during quit: {e} - connection was likely already closed.")

    ftp.quit()
    
    # now read in the result (cannot remember why I did this in two steps)
    metadata_aboutSamples = pd.read_table(writeFile,delimiter = '\t')
    
    # pull what I can from the sample information at MetaboLights
    #for the database this is all need (is there a sample...)
    sampleNames  = metadata_aboutSamples['Source Name']

    #MetaboLights required samples to begin with a letter, I used 's' and need to strip that out 
    NewID_inMTBLS  = pd.to_numeric(sampleNames.str.strip('s')) 

    #convert the series into a dataframe:
    df = NewID_inMTBLS.reset_index() 
        
    #%run Kuj_MetabolightsData.py
    
    session = Session()
    #pdb.set_trace()
    for index,row in tqdm(df.iterrows()):
        db = MetaboliteInfo()
        db.bottleID = f"{row['Source Name']}"
        db.dataSource = study_id 
        session.add(db)
    session.commit()

def load_metaboliteUntargeted_info():
    print("Loading metabolite (untargeted) information from MetaboLights")
    dataDir = '../test_data/'
    # start with one dataset at MetaboLights --> MTBLS5228 is McParland et al.
    study_id = 'MTBLS5228'
    try:
        ftp = FTP('ftp.ebi.ac.uk') #address from MetaboLights webpage
        ftp.login()
        ftpDataAddress = '/pub/databases/metabolights/studies/public/' + study_id
        ftp.cwd(ftpDataAddress)
        fileList = ftp.nlst() #can use this to make a list that will be searchable
        
        #start with the metadata about the samples 
        str = 's_' + study_id #this is the search string for the data files
        metadataFiles = [v for v in fileList if str in v] 
        metadataFiles = pd.DataFrame(metadataFiles,columns = ['files'])
        readFile = metadataFiles.loc[0,'files']

        #pdb.set_trace()
        # metadataFiles: put them here 
        # Is there a way to download an FTP file and not write it disk?
        #writeFile = 'dataDir/' + 'tempMetadata.txt'
        writeFile = os.path.join(dataDir,'tempMetadata.txt')

        with open(writeFile,'wb') as fp:
            try:
                retr_command = f"RETR {readFile}"
                ftp.retrbinary(retr_command, fp.write)
            except Exception as e: 
                print(f"Error during quit: {e}")
            except AttributeError as e: 
                print(f"AttributeError during quit: {e} - connection was likely already closed.")

        ftp.quit()
        
        # now read in the result (cannot remember why I did this in two steps)
        metadata_aboutSamples = pd.read_table(writeFile,delimiter = '\t')
        
        # pull what I can from the sample information at MetaboLights
        #for the database this is all need (is there a sample...)
        sampleNames  = metadata_aboutSamples['Source Name']

        #Erin also had blanks and boooled samples, remove those and make a list (added 4/6/2026)
        #Don't really want the extra steps, but that is how I can understand this
        sampleList = [x for x in sampleNames if not x.startswith(('spool','sblank','smqblank'))]
        sampleNames = pd.Series(sampleList)
        
        #MetaboLights required samples to begin with a letter, I used 's' and need to strip that out 
        NewID_inMTBLS  = pd.to_numeric(sampleNames.str.strip('s')) 

        #convert the series into a dataframe:
        df = NewID_inMTBLS.reset_index() 
        df.columns = ['index','bottleID'] #somehow lost column labels
                   
        session = Session()
        for index,row in tqdm(df.iterrows()):
            #pdb.set_trace()
            db = MtabUntargetedInfo()
            db.bottleID = f"{row['bottleID']}" #seems like there should be a btter way to do this
            db.dataSource = study_id 
            session.add(db)
        session.commit()
    except:
        print("MetaboLights did not allow connection, dummy data OR some other error")
        session = Session()
        #pdb.set_trace()
        df = pd.DataFrame({'bottleID':['1033900707'],'dataSource':['MetaboLightsNotAvailable']})
        for index,row in tqdm(df.iterrows()):
            db = MtabUntargetedInfo()
            db.bottleID = f"{row['bottleID']}" 
            db.dataSource = row['dataSource']
            session.add(db)
        session.commit()  

In [18]:
#now run the functions
load_V416S_sequencing_info()
load_V1V2_sequencing_info()
load_discrete_info()
load_metabolite_info()
load_metaboliteUntargeted_info()
load_cyverse_info()

Loading V416S sequencing information


2064it [00:00, 3261.43it/s]


Loading V1V2 sequencing information


FileNotFoundError: [Errno 2] No such file or directory: '../test_data/BIOS-SCOPE time series/V1V2_dada2_read_info_03052026.xlsx'

In [62]:
from sqlalchemy import inspect
inspector = inspect(engine)
print(inspector.get_table_names())

['cyverse', 'discrete', 'metabolites', 'metabolitesUntargeted', 'sequencingV1V2', 'sequencingV4']


In [63]:
from sqlalchemy import create_engine, inspect, MetaData, Table
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)

#reflect the tables so I can work on them
user_seqV4 = Table('sequencingV4', metadata_obj, autoload_with=engine)
user_seqV1V2 = Table('sequencingV1V2', metadata_obj, autoload_with=engine)
user_cy = Table('cyverse',metadata_obj,autoload_with=engine)
user_discrete = Table('discrete',metadata_obj,autoload_with=engine)
user_mtab = Table('metabolites',metadata_obj,autoload_with=engine)
user_mtabUntargeted = Table('metabolitesUntargeted',metadata_obj,autoload_with=engine)

user_cy

Table('cyverse', MetaData(), Column('id', INTEGER(), table=<cyverse>, primary_key=True, nullable=False), Column('filename', VARCHAR(), table=<cyverse>), Column('source', VARCHAR(), table=<cyverse>), schema=None)

### update the database (based on query results)
Start with the V4 data

In [ ]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

# #see all of what is in table
session.query(SeqInfoV4).all()

In [66]:
#updating...
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV4.c.filename)
    .where(user_discrete.c.bottleID == user_seqV4.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V4data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()    

In [67]:
session.query(DiscreteInfo).all()

[<DiscreteInfo(bottleID='1033900707', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900708', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900709', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900710', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900711', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900712', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900713', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900714', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900715', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900716', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900717', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900718', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900719', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900720', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900721', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900722', cruise='AE1718')>,
 <DiscreteInfo(bottleID='1033900723', cruise='AE1718')>,
 <DiscreteInfo(bottleID='103390

In [68]:
# see if this worked
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('discrete', metadata,
    Column('id', Integer, primary_key=True),
    Column('bottleID', String),
    Column('cruise', String),
    Column('yyyymmdd',String),
    Column('V4data',String)              
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)
    df = df.reindex(columns = ['bottleID','cruise','yyyymmdd','V4data'])

df.head()
df.to_csv('temp.csv')

In [122]:
#move on to V1V2 (surely there is a better way to do this...)

In [123]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

In [124]:
# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV1V2.c.V1V2data)
    .where(user_discrete.c.bottleID == user_seqV1V2.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V1V2data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

In [ ]:
#now get more complicated...instead of the name of the file with data, do we
#find the sequencing data at cyverse/google...BUT...will have different ones for each
#sequencing type...so what is the best way to do this?

In [45]:
user_seqV4

Table('sequencingV4', MetaData(), Column('id', INTEGER(), table=<sequencingV4>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<sequencingV4>), Column('cast', VARCHAR(), table=<sequencingV4>), Column('NominalDepth', VARCHAR(), table=<sequencingV4>), Column('filename', VARCHAR(), table=<sequencingV4>), Column('V4data', VARCHAR(), table=<sequencingV4>), schema=None)

In [15]:
user_cy

Table('cyverse', MetaData(), Column('id', INTEGER(), table=<cyverse>, primary_key=True, nullable=False), Column('filename', VARCHAR(), table=<cyverse>), Column('source', VARCHAR(), table=<cyverse>), schema=None)

In [ ]:
#can I match the sequencing file in cyverse/google with the information in DNA master?
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()


# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV4.c.V4data)
    .where(user_discrete.c.bottleID == user_seqV4.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V4data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()  

In [126]:
#finally the metabolite data
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_mtab.c.dataSource)
    .where(user_discrete.c.bottleID == user_mtab.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(mtabData=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

In [127]:
session.query(user_mtabUntargeted).all()

[(1, '1033001402', 'MTBLS5228'),
 (2, '1033001405', 'MTBLS5228'),
 (3, '1033001410', 'MTBLS5228'),
 (4, '1033001414', 'MTBLS5228'),
 (5, '1033001420', 'MTBLS5228'),
 (6, '1033001424', 'MTBLS5228'),
 (7, '1033201501', 'MTBLS5228'),
 (8, '1033201504', 'MTBLS5228'),
 (9, '1033201507', 'MTBLS5228'),
 (10, '1033201511', 'MTBLS5228'),
 (11, '1033201518', 'MTBLS5228'),
 (12, '1033201523', 'MTBLS5228'),
 (13, '1033500702', 'MTBLS5228'),
 (14, '1033500705', 'MTBLS5228'),
 (15, '1033500710', 'MTBLS5228'),
 (16, '1033500714', 'MTBLS5228'),
 (17, '1033500720', 'MTBLS5228'),
 (18, '1033500724', 'MTBLS5228'),
 (19, '1033901102', 'MTBLS5228'),
 (20, '1033901109', 'MTBLS5228'),
 (21, '1033901120', 'MTBLS5228'),
 (22, '1034101601', 'MTBLS5228'),
 (23, '1034101603', 'MTBLS5228'),
 (24, '1034101607', 'MTBLS5228'),
 (25, '1034101611', 'MTBLS5228'),
 (26, '1034101617', 'MTBLS5228'),
 (27, '1034101623', 'MTBLS5228'),
 (28, '1034301202', 'MTBLS5228'),
 (29, '1034301204', 'MTBLS5228'),
 (30, '1034301208', 'MT

In [49]:
# see if this worked
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('discrete', metadata,
    Column('id', Integer, primary_key=True),
    Column('bottleID', String),
    Column('cruise', String),
    Column('yyyymmdd',String),
    Column('V4data',String),
    Column('V1V2data',String),
    Column('mtabData',String),
    Column('mtabDataUntargeted',String)
              
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)
    df = df.reindex(columns = ['bottleID','cruise','yyyymmdd','V1V2data','V4data','mtabData','mtabDataUntargeted'])

df.head()

OperationalError: (sqlite3.OperationalError) no such column: discrete.mtabDataUntargeted
[SQL: SELECT discrete.id, discrete."bottleID", discrete.cruise, discrete.yyyymmdd, discrete."V4data", discrete."V1V2data", discrete."mtabData", discrete."mtabDataUntargeted" 
FROM discrete]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [76]:
#%run populate_db.py

Exception: File `'populate_db.py'` not found.

In [ ]:
#Stick some code below this spot as a holding zone

raise SystemExit("Stop execution here")

In [ ]:
user_table_seqInfo.primary_key

In [ ]:
from typing import List
from typing import Optional
from sqlalchemy import Column, String
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class TestingNew(Base):
    __tablename__ = 'testingNew'
    id: Mapped[int] = mapped_column(primary_key=True)
    bottleID: Mapped[int] = mapped_column(String(10))
    cruise: Mapped[str] = mapped_column(String(10))
    cast: Mapped[int] = mapped_column(String(10))
    niskin: Mapped[int] = mapped_column(String(10))
    nominalDepth: Mapped[int] = mapped_column(String(10))
    
# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"